# Week 4 — PhoBERT-Guided LLM Verifier

Pipeline final:
1. Pull code moi nhat tu GitHub.
2. Lay checkpoint PhoBERT da train tu repo hoac Kaggle Dataset.
3. Chay PhoBERT argmax baseline.
4. Chay PhoBERT-Guided LLM Verifier: PhoBERT sinh candidate + confidence, LLM chi verify/rerank cac case uncertain.

Mac dinh notebook nay **khong train lai PhoBERT** va **khong chay lai ADD-only/SPC cu**.

In [ ]:
# Cell 1 — Kaggle GPU + dependencies
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
else:
    raise RuntimeError('Khong co GPU. Kaggle: Settings -> Accelerator -> GPU T4')

torch.cuda.empty_cache()
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

!pip install -q --upgrade pip
!pip install -q transformers==4.41.2 sentence-transformers==2.7.0 tokenizers==0.19.1 \
    accelerate==0.30.1 underthesea py_vncorenlp tabulate tqdm scikit-learn sentencepiece faiss-cpu
!pip install -q openai google-generativeai
print('Dependencies installed')

In [ ]:
# Cell 2 — Clone/pull latest repo
import os, sys

REPO_URL = 'https://github.com/vudinhminh08/NLP-project-master-study.git'
REPO_BRANCH = 'master'
PROJECT_DIR = '/kaggle/working/absa-project'

if not os.path.exists(PROJECT_DIR):
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

for p in ['code/week1', 'code/week2', 'code/week3', 'code/week3_part2', 'code/week4']:
    full = os.path.join(PROJECT_DIR, p)
    if full not in sys.path:
        sys.path.insert(0, full)

print('sys.path updated')
!git rev-parse --short HEAD

In [ ]:
# Cell 3 — Load API key from Kaggle Secrets
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# Recommended for this run
LLM_PROVIDER = 'openai'
LLM_API_KEY = secrets.get_secret('OPENAI_API_KEY')
print('OPENAI_API_KEY loaded')

# Optional Gemini switch:
# LLM_PROVIDER = 'gemini'
# LLM_API_KEY = secrets.get_secret('GEMINI_API_KEY')

In [ ]:
# Cell 4 — Locate/copy checkpoint, data, and optional embeddings cache
import os, shutil
from glob import glob

CHECKPOINT_PATH = 'outputs/results/week2_results_VNcoreNLP/models_cls_only/best_model.pt'
EMBEDDINGS_CACHE = 'outputs/results/embeddings_cache.npy'

REQUIRED_FILES = [
    'data/train_preprocessed.csv',
    'data/dev_preprocessed.csv',
    'data/test_preprocessed.csv',
    'outputs/eda/class_weights.json',
]

def copy_first_match(patterns, dst, required=False):
    if os.path.exists(dst):
        print(f'[OK] {dst}')
        return dst
    for pat in patterns:
        matches = glob(pat, recursive=True)
        matches = [m for m in matches if os.path.isfile(m)]
        if matches:
            os.makedirs(os.path.dirname(dst) or '.', exist_ok=True)
            shutil.copy(matches[0], dst)
            print(f'[COPIED] {matches[0]} -> {dst}')
            return dst
    if required:
        raise FileNotFoundError(f'Missing required file: {dst}')
    print(f'[OPTIONAL MISSING] {dst}')
    return None

# best_model.pt: upload vao Kaggle Dataset neu file qua nang de dua len Git.
copy_first_match([
    '/kaggle/input/**/week2_results_VNcoreNLP/models_cls_only/best_model.pt',
    '/kaggle/input/**/models_cls_only/best_model.pt',
    '/kaggle/input/**/best_model.pt',
], CHECKPOINT_PATH, required=True)

# Data/config: usually in repo, fallback to Kaggle Dataset.
for dst in REQUIRED_FILES:
    copy_first_match([
        f'/kaggle/input/**/{os.path.basename(dst)}',
        f'/kaggle/input/**/{dst}',
    ], dst, required=True)

# Embeddings cache optional. If absent, retriever will recompute train embeddings.
copy_first_match([
    '/kaggle/input/**/embeddings_cache.npy',
    '/kaggle/input/**/outputs/results/embeddings_cache.npy',
], EMBEDDINGS_CACHE, required=False)

print('File verification done')

In [ ]:
# Cell 5 — Load data and dataloaders
import json
import pandas as pd
from transformers import AutoTokenizer

from utils.constants import PHOBERT_V2, TRAIN_CONFIG, ZERO_TRAIN_ASPECTS
from utils.helpers import set_seed
from step2_dataloader import create_dataloaders

set_seed(TRAIN_CONFIG['seed'])

train_df = pd.read_csv('data/train_preprocessed.csv')
dev_df = pd.read_csv('data/dev_preprocessed.csv')
test_df = pd.read_csv('data/test_preprocessed.csv')
class_weights = json.load(open('outputs/eda/class_weights.json'))

tokenizer = AutoTokenizer.from_pretrained(PHOBERT_V2)
_, dev_loader, test_loader = create_dataloaders(
    train_path='data/train_preprocessed.csv',
    dev_path='data/dev_preprocessed.csv',
    test_path='data/test_preprocessed.csv',
    tokenizer=tokenizer,
    batch_size=32,
    use_preprocessed=True,
)

print(f'train={len(train_df)} | dev={len(dev_df)} | test={len(test_df)}')
print(f'checkpoint={CHECKPOINT_PATH}')

In [ ]:
# Cell 6 — Smoke test: 10 reviews, SPC-first guided verifier
from run_week4_experiment import run_week4, Week4Config

smoke_config = Week4Config(
    checkpoint_path=CHECKPOINT_PATH,
    output_dir='outputs/results/week4_guided_verifier_spc_smoke',
    embeddings_cache=EMBEDDINGS_CACHE,
    llm_provider=LLM_PROVIDER,
    llm_api_key=LLM_API_KEY,
    run_add_only_variant=False,
    run_add_spc_variant=False,
    run_guided_verifier=True,
    max_test_samples=10,
    guided_enable_add=False,
    guided_enable_spc=True,
    guided_add_threshold=0.08,
    guided_spc_entropy_threshold=0.55,
    guided_k_rag=8,
    guided_max_candidates_per_review=8,
    guided_delete_enabled=False,
    guided_require_evidence_for_spc=False,
    guided_apply_label_prior=True,
    sleep_sec=0.3,
)

smoke_results = run_week4(
    config=smoke_config,
    train_df=train_df,
    dev_df=dev_df,
    test_df=test_df,
    dev_loader=dev_loader,
    test_loader=test_loader,
    class_weights=class_weights,
)

smoke_results


In [ ]:
# Cell 7 — Full test run: set RUN_FULL=True after smoke test is OK
RUN_FULL = False

if RUN_FULL:
    full_config = Week4Config(
        checkpoint_path=CHECKPOINT_PATH,
        output_dir='outputs/results/week4_guided_verifier_spc',
        embeddings_cache=EMBEDDINGS_CACHE,
        llm_provider=LLM_PROVIDER,
        llm_api_key=LLM_API_KEY,
        run_add_only_variant=False,
        run_add_spc_variant=False,
        run_guided_verifier=True,
        max_test_samples=None,
        guided_enable_add=False,
        guided_enable_spc=True,
        guided_add_threshold=0.08,
        guided_spc_entropy_threshold=0.55,
        guided_k_rag=8,
        guided_max_candidates_per_review=8,
        guided_delete_enabled=False,
        guided_require_evidence_for_spc=False,
        guided_apply_label_prior=True,
        sleep_sec=0.3,
    )

    full_results = run_week4(
        config=full_config,
        train_df=train_df,
        dev_df=dev_df,
        test_df=test_df,
        dev_loader=dev_loader,
        test_loader=test_loader,
        class_weights=class_weights,
    )
    print(full_results)
else:
    print('RUN_FULL=False. Set RUN_FULL=True after smoke test passes.')


In [ ]:
# Cell 8 — Show saved metrics/stats
import json, os

RESULT_DIR = 'outputs/results/week4_guided_verifier_spc'
SMOKE_DIR = 'outputs/results/week4_guided_verifier_spc_smoke'

def show_json(path):
    if not os.path.exists(path):
        print(f'[missing] {path}')
        return None
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    print(f'\n=== {path} ===')
    print(json.dumps(data, ensure_ascii=False, indent=2)[:4000])
    return data

show_json(os.path.join(SMOKE_DIR, 'guided_verifier_metrics.json'))
show_json(os.path.join(SMOKE_DIR, 'guided_verifier_stats.json'))
show_json(os.path.join(RESULT_DIR, 'guided_verifier_metrics.json'))
show_json(os.path.join(RESULT_DIR, 'guided_verifier_stats.json'))